# Fixed RAG Notebook: 3 PDFs → OpenAI Embeddings → Pinecone → OpenAI Chat

This notebook is a cleaned and runnable version of the original RAG pipeline.

## What was fixed

- Removed the deprecated / conflicting Pinecone import path issues.
- Removed the Hugging Face embedding path because this version uses OpenAI embeddings.
- Removed duplicate imports.
- Removed hard-coded `os.chdir(...)` to a local Windows path.
- Avoided the old `pkg_resources` environment dump.
- Added Pinecone index dimension checks.
- Added stable chunk IDs and metadata.
- Added a custom Pinecone retriever, so this notebook does **not** require `langchain-pinecone`.
- Removed the broken Streamlit cell that used an undefined `user_question`.


## 0. One-time environment repair

If you previously installed `pinecone-client`, uninstall it. The official Pinecone package is now `pinecone`.

Run the next cell once if your imports fail, then restart the notebook kernel and run the notebook again.


In [1]:
# RUN THIS CELL ONLY IF YOUR ENVIRONMENT / IMPORTS ARE BROKEN.
# After running it, restart the kernel.

RUN_ENV_REPAIR = False

if RUN_ENV_REPAIR:
    import sys
    import subprocess

    commands = [
        [sys.executable, "-m", "pip", "uninstall", "-y", "pinecone-client"],
        [
            sys.executable, "-m", "pip", "install", "-U",
            "pinecone",
            "openai",
            "langchain",
            "langchain-community",
            "langchain-core",
            "langchain-text-splitters",
            "langchain-openai",
            "pypdf",
            "python-dotenv",
            "pydantic",
        ],
    ]

    for cmd in commands:
        print("Running:", " ".join(cmd))
        subprocess.check_call(cmd)

    print("\nEnvironment repair complete. Restart the kernel before continuing.")
else:
    print("Skipping environment repair. Set RUN_ENV_REPAIR = True only if imports fail.")

Skipping environment repair. Set RUN_ENV_REPAIR = True only if imports fail.


## 1. Imports


In [2]:
# Core Python
import os
import time
import hashlib
import warnings
import importlib.metadata as metadata
from pathlib import Path
from typing import Any, Dict, List, Optional

# Environment variables
from dotenv import load_dotenv

# Pinecone SDK
from pinecone import Pinecone, ServerlessSpec

# LangChain: document loading and splitting
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain: OpenAI models and embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangChain: custom retriever and RAG chain components
from pydantic import PrivateAttr
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Custom chain wrapper classes for LangChain 1.2.17+
class StuffDocumentsChain:
    """Custom implementation of create_stuff_documents_chain with invoke method"""
    
    def __init__(self, llm, prompt, document_prompt=None):
        self.llm = llm
        self.prompt = prompt
        self.document_prompt = document_prompt
    
    def format_docs(self, docs):
        if self.document_prompt:
            formatted = []
            for doc in docs:
                formatted.append(self.document_prompt.format(**doc.metadata, page_content=doc.page_content))
            return "\n\n".join(formatted)
        else:
            return "\n\n".join(doc.page_content for doc in docs)
    
    def invoke(self, inputs):
        docs = inputs["context"]
        context = self.format_docs(docs)
        # Create a new dict with the formatted context, avoiding duplicate keys
        prompt_inputs = {k: v for k, v in inputs.items() if k != "context"}
        prompt_inputs["context"] = context
        messages = self.prompt.format_messages(**prompt_inputs)
        response = self.llm.invoke(messages)
        return {"answer": response.content, "context": docs}

class RetrievalChain:
    """Custom implementation of create_retrieval_chain with invoke method"""
    
    def __init__(self, retriever, combine_docs_chain):
        self.retriever = retriever
        self.combine_docs_chain = combine_docs_chain
    
    def invoke(self, inputs):
        # Get the query
        query = inputs["input"]
        
        # Retrieve relevant documents
        docs = self.retriever.invoke(query)
        
        # Pass to the combine chain
        chain_input = {"context": docs, "input": query}
        result = self.combine_docs_chain.invoke(chain_input)
        
        return result

def create_stuff_documents_chain(llm, prompt, document_prompt=None):
    """Custom implementation of create_stuff_documents_chain"""
    return StuffDocumentsChain(llm, prompt, document_prompt)

def create_retrieval_chain(retriever, combine_docs_chain):
    """Custom implementation of create_retrieval_chain"""
    return RetrievalChain(retriever, combine_docs_chain)

warnings.filterwarnings("default")

print("Imports completed successfully with custom chain implementations.")

c:\Users\omarc\Data Science Repositories\rag-ai-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports completed successfully with custom chain implementations.


## 2. Check key package versions

This checks only the packages that matter for this notebook.


In [3]:
packages_to_check = [
    "pinecone",
    "openai",
    "langchain",
    "langchain-community",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-openai",
    "pypdf",
    "python-dotenv",
    "pydantic",
]

for package in packages_to_check:
    try:
        print(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


pinecone==7.3.0
openai==2.36.0
langchain==1.2.17
langchain-community==0.4.1
langchain-core==1.3.3
langchain-text-splitters==1.1.2
langchain-openai==1.2.1
pypdf==6.10.2
python-dotenv==1.2.2
pydantic==2.13.4


## 3. Project paths and environment variables

Expected project structure:

```text
rag-ai-pipeline/
├── data/
│   └── external/
│       ├── document_1.pdf
│       ├── document_2.pdf
│       └── document_3.pdf
├── env/
│   └── api_keys.env
└── notebooks/
    └── main_fixed_rag_openai_pinecone.ipynb
```

Your `api_keys.env` file should contain:

```text
OPENAI_API_KEY=your_openai_key
PINECONE_API_KEY=your_pinecone_key
```


In [4]:
# Robust project-root detection.
# This lets the notebook work whether it is opened from the repo root or from a notebooks/ folder.

cwd = Path.cwd().resolve()

candidate_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / "data").exists() or (candidate / "env").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd

PDF_DIR = PROJECT_ROOT / "data" / "external"

candidate_env_paths = [
    PROJECT_ROOT / "env" / "api_keys.env",
    PROJECT_ROOT / ".env",
    cwd / "env" / "api_keys.env",
    cwd / ".env",
]

loaded_env_path = None
for env_path in candidate_env_paths:
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
        loaded_env_path = env_path
        break

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

print(f"Current working directory:  {cwd}")
print(f"Detected project root:      {PROJECT_ROOT}")
print(f"PDF directory:              {PDF_DIR}")
print(f"Loaded env file:            {loaded_env_path if loaded_env_path else 'No env file found; using existing environment variables'}")

missing_keys = [key for key, value in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "PINECONE_API_KEY": PINECONE_API_KEY,
}.items() if not value]

if missing_keys:
    raise EnvironmentError(
        f"Missing required environment variable(s): {missing_keys}. "
        "Add them to env/api_keys.env or your system environment."
    )

# Make sure downstream libraries can read the key from the environment.
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

print("API keys loaded successfully.")


Current working directory: C:\Users\omarc\Data Science Repositories\rag-ai-pipeline\main
Detected project root:      C:\Users\omarc\Data Science Repositories\rag-ai-pipeline
PDF directory:              C:\Users\omarc\Data Science Repositories\rag-ai-pipeline\data\external
Loaded env file:            C:\Users\omarc\Data Science Repositories\rag-ai-pipeline\env\api_keys.env
API keys loaded successfully.


## 4. Define model and Pinecone settings

This notebook uses OpenAI embeddings, so the Pinecone index dimension must match the embedding model.

- `text-embedding-3-small` → 1536 dimensions
- `text-embedding-3-large` → 3072 dimensions

This notebook defaults to `text-embedding-3-small`.


In [5]:
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536

CHAT_MODEL = "gpt-4o-mini"

# Use a new index name to avoid dimension conflicts with older Hugging Face 384-dim indexes.
INDEX_NAME = "rag-3-pdf-openai"

PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"

# Optional namespace keeps this project isolated inside the index.
NAMESPACE = "personal-finance-pdfs"

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Embedding dimension: {EMBEDDING_DIMENSION}")
print(f"Chat model: {CHAT_MODEL}")
print(f"Pinecone index: {INDEX_NAME}")
print(f"Pinecone namespace: {NAMESPACE}")


Embedding model: text-embedding-3-small
Embedding dimension: 1536
Chat model: gpt-4o-mini
Pinecone index: rag-3-pdf-openai
Pinecone namespace: personal-finance-pdfs


## 5. Load the PDF documents


In [6]:
def list_pdf_files(pdf_dir: Path) -> List[Path]:
    if not pdf_dir.exists():
        raise FileNotFoundError(
            f"PDF directory does not exist: {pdf_dir}\n"
            "Create this folder and place your 3 PDF files inside it."
        )

    pdf_files = sorted(pdf_dir.glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in: {pdf_dir}")

    print(f"Found {len(pdf_files)} PDF file(s):")
    for pdf in pdf_files:
        print(f" - {pdf.name}")

    if len(pdf_files) != 3:
        print(
            "\nWarning: this project was designed for exactly 3 PDFs, "
            f"but found {len(pdf_files)}. The notebook will continue using all PDFs found."
        )

    return pdf_files


pdf_files = list_pdf_files(PDF_DIR)


Found 3 PDF file(s):
 - personalfinancefordummies.pdf
 - richdadpoordad.pdf
 - theintelligentinvestorbook.pdf


In [7]:
def load_pdf_files(pdf_dir: Path) -> List[Document]:
    loader = DirectoryLoader(
        str(pdf_dir),
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True,
    )

    documents = loader.load()

    # Normalize source path to file name for cleaner citations.
    for doc in documents:
        source = doc.metadata.get("source", "")
        doc.metadata["source"] = Path(source).name if source else "unknown_source"
        doc.metadata["page"] = int(doc.metadata.get("page", 0)) + 1  # convert to 1-based page numbers

    print(f"Loaded {len(documents)} page-level document(s).")
    return documents


raw_documents = load_pdf_files(PDF_DIR)

# Preview one loaded page
print("\nPreview metadata:")
print(raw_documents[0].metadata)
print("\nPreview content:")
print(raw_documents[0].page_content[:500])


100%|██████████| 3/3 [00:07<00:00,  2.43s/it]

Loaded 1365 page-level document(s).

Preview metadata:
{'producer': 'Acrobat Distiller 6.0.1 (Windows)', 'creator': 'Adobe InDesign CS4 (6.0.2)', 'creationdate': '2009-10-01T00:15:41-04:00', 'author': 'Tyson, Eric.', 'moddate': '2013-10-14T22:02:30-04:00', 'title': 'Personal Finance for Dummies', 'source': 'personalfinancefordummies.pdf', 'total_pages': 483, 'page': 1, 'page_label': '1'}

Preview content:
Eric Tyson, MBA
Bestselling author, Investing For Dummies
Learn to:
   Assess your financial fitness

 Save more and spend less

  Review your credit report and improve 
your
 score

  Make smart investments in any 
economic en
vironment
Personal 
Finance
6th EditionMaking Everything Easier!
™
“Provides tremendous insight and guidance into 
the world of investing and other money issues.”
                                                                                            – PBS Nightly


## 6. Split documents into chunks

Each PDF page is split into smaller chunks so Pinecone can retrieve more focused evidence.


In [8]:
def split_documents(
    documents: List[Document],
    chunk_size: int = 800,
    chunk_overlap: int = 150,
) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = text_splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):
        source = chunk.metadata.get("source", "unknown_source")
        page = chunk.metadata.get("page", "unknown_page")
        content_hash = hashlib.sha1(chunk.page_content.encode("utf-8")).hexdigest()[:12]

        chunk.metadata["chunk_id"] = f"{Path(str(source)).stem}-p{page}-c{i}-{content_hash}"
        chunk.metadata["source_file"] = source

    print(f"Created {len(chunks)} text chunk(s).")
    return chunks

text_chunks = split_documents(raw_documents)

print("\nPreview chunk metadata:")
print(text_chunks[0].metadata)
print("\nPreview chunk content:")
print(text_chunks[0].page_content[:500])


Created 4202 text chunk(s).

Preview chunk metadata:
{'producer': 'Acrobat Distiller 6.0.1 (Windows)', 'creator': 'Adobe InDesign CS4 (6.0.2)', 'creationdate': '2009-10-01T00:15:41-04:00', 'author': 'Tyson, Eric.', 'moddate': '2013-10-14T22:02:30-04:00', 'title': 'Personal Finance for Dummies', 'source': 'personalfinancefordummies.pdf', 'total_pages': 483, 'page': 1, 'page_label': '1', 'chunk_id': 'personalfinancefordummies-p1-c0-8c75574a78cf', 'source_file': 'personalfinancefordummies.pdf'}

Preview chunk content:
Eric Tyson, MBA
Bestselling author, Investing For Dummies
Learn to:
   Assess your financial fitness

 Save more and spend less

  Review your credit report and improve 
your
 score

  Make smart investments in any 
economic en
vironment
Personal 
Finance
6th EditionMaking Everything Easier!
™
“Provides tremendous insight and guidance into 
the world of investing and other money issues.”
                                                                                    

## 7. Initialize OpenAI embeddings and Pinecone


In [9]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

pc = Pinecone(api_key=PINECONE_API_KEY)

print("OpenAI embeddings and Pinecone client initialized.")


OpenAI embeddings and Pinecone client initialized.


In [10]:
def get_pinecone_index_names(pc: Pinecone) -> List[str]:
    indexes = pc.list_indexes()

    # Newer SDKs often expose .names()
    if hasattr(indexes, "names"):
        return indexes.names()

    # Fallback for iterable index objects / dictionaries
    names = []
    for idx in indexes:
        if isinstance(idx, dict):
            names.append(idx.get("name"))
        else:
            names.append(getattr(idx, "name", None))

    return [name for name in names if name]


def create_pinecone_index_if_needed(
    pc: Pinecone,
    index_name: str,
    dimension: int,
    metric: str = "cosine",
    cloud: str = "aws",
    region: str = "us-east-1",
) -> None:
    existing_index_names = get_pinecone_index_names(pc)

    if index_name in existing_index_names:
        print(f"Using existing Pinecone index: {index_name}")

        # Validate dimension if the SDK exposes describe_index.
        try:
            description = pc.describe_index(index_name)
            existing_dimension = getattr(description, "dimension", None)

            if existing_dimension is None and isinstance(description, dict):
                existing_dimension = description.get("dimension")

            if existing_dimension is not None and int(existing_dimension) != int(dimension):
                raise ValueError(
                    f"Existing index '{index_name}' has dimension {existing_dimension}, "
                    f"but embedding model '{EMBEDDING_MODEL}' requires dimension {dimension}. "
                    "Use a new index name or delete/recreate the existing index."
                )
        except AttributeError:
            # Older/newer SDK surface difference; skip dimension validation.
            pass

        return

    print(f"Creating Pinecone index: {index_name}")

    spec = ServerlessSpec(cloud=cloud, region=region)

    # Most Pinecone SDK versions support pc.create_index.
    if hasattr(pc, "create_index"):
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=spec,
        )
    else:
        # Fallback for SDKs exposing the nested indexes client.
        pc.indexes.create(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=spec,
        )

    # Wait until ready.
    for _ in range(60):
        try:
            description = pc.describe_index(index_name)
            status = getattr(description, "status", None)

            if isinstance(status, dict) and status.get("ready"):
                print("Index is ready.")
                return

            if getattr(status, "ready", False):
                print("Index is ready.")
                return

        except Exception:
            pass

        time.sleep(2)

    print("Index creation requested. If the next cell fails, wait a minute and rerun it.")


create_pinecone_index_if_needed(
    pc=pc,
    index_name=INDEX_NAME,
    dimension=EMBEDDING_DIMENSION,
    metric="cosine",
    cloud=PINECONE_CLOUD,
    region=PINECONE_REGION,
)

index = pc.Index(INDEX_NAME)

print("Connected to Pinecone index.")


Using existing Pinecone index: rag-3-pdf-openai
Connected to Pinecone index.


## 8. Upsert chunks into Pinecone

This embeds each chunk using OpenAI and stores the vector plus metadata in Pinecone.


In [11]:
def clean_metadata(metadata: Dict[str, Any]) -> Dict[str, Any]:
    """
    Pinecone metadata values must be simple types:
    string, number, boolean, or list of strings.
    """
    cleaned = {}

    for key, value in metadata.items():
        if value is None:
            continue

        if isinstance(value, (str, int, float, bool)):
            cleaned[key] = value
        elif isinstance(value, list):
            cleaned[key] = [str(item) for item in value]
        else:
            cleaned[key] = str(value)

    return cleaned

def upsert_documents_to_pinecone(
    documents: List[Document],
    embedding_model: OpenAIEmbeddings,
    pinecone_index: Any,
    namespace: Optional[str] = None,
    batch_size: int = 100,
) -> None:
    total = len(documents)

    for batch_start in range(0, total, batch_size):
        batch = documents[batch_start: batch_start + batch_size]
        texts = [doc.page_content for doc in batch]
        vectors = embedding_model.embed_documents(texts)

        upsert_payload = []

        for doc, vector in zip(batch, vectors):
            chunk_id = doc.metadata["chunk_id"]
            metadata = clean_metadata({
                **doc.metadata,
                "text": doc.page_content,
            })

            upsert_payload.append({
                "id": chunk_id,
                "values": vector,
                "metadata": metadata,
            })

        kwargs = {"vectors": upsert_payload}
        if namespace:
            kwargs["namespace"] = namespace

        pinecone_index.upsert(**kwargs)

        print(f"Upserted {min(batch_start + batch_size, total)} / {total} chunks")

    print("Finished upserting documents to Pinecone.")


upsert_documents_to_pinecone(
    documents=text_chunks,
    embedding_model=embeddings,
    pinecone_index=index,
    namespace=NAMESPACE,
    batch_size=100,
)


Upserted 100 / 4202 chunks
Upserted 200 / 4202 chunks
Upserted 300 / 4202 chunks
Upserted 400 / 4202 chunks
Upserted 500 / 4202 chunks
Upserted 600 / 4202 chunks
Upserted 700 / 4202 chunks
Upserted 800 / 4202 chunks
Upserted 900 / 4202 chunks
Upserted 1000 / 4202 chunks
Upserted 1100 / 4202 chunks
Upserted 1200 / 4202 chunks
Upserted 1300 / 4202 chunks
Upserted 1400 / 4202 chunks
Upserted 1500 / 4202 chunks
Upserted 1600 / 4202 chunks
Upserted 1700 / 4202 chunks
Upserted 1800 / 4202 chunks
Upserted 1900 / 4202 chunks
Upserted 2000 / 4202 chunks
Upserted 2100 / 4202 chunks
Upserted 2200 / 4202 chunks
Upserted 2300 / 4202 chunks
Upserted 2400 / 4202 chunks
Upserted 2500 / 4202 chunks
Upserted 2600 / 4202 chunks
Upserted 2700 / 4202 chunks
Upserted 2800 / 4202 chunks
Upserted 2900 / 4202 chunks
Upserted 3000 / 4202 chunks
Upserted 3100 / 4202 chunks
Upserted 3200 / 4202 chunks
Upserted 3300 / 4202 chunks
Upserted 3400 / 4202 chunks
Upserted 3500 / 4202 chunks
Upserted 3600 / 4202 chunks
U

## 9. Build a custom Pinecone retriever

This avoids the `langchain_pinecone` dependency entirely while still integrating with LangChain's RAG chain interface.


In [12]:
class PineconeRetriever(BaseRetriever):
    k: int = 5
    namespace: Optional[str] = None
    search_filter: Optional[Dict[str, Any]] = None

    _index: Any = PrivateAttr()
    _embeddings: Any = PrivateAttr()

    def __init__(
        self,
        index: Any,
        embeddings: OpenAIEmbeddings,
        k: int = 5,
        namespace: Optional[str] = None,
        search_filter: Optional[Dict[str, Any]] = None,
        **kwargs: Any,
    ):
        super().__init__(
            k=k,
            namespace=namespace,
            search_filter=search_filter,
            **kwargs,
        )
        self._index = index
        self._embeddings = embeddings

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        query_vector = self._embeddings.embed_query(query)

        query_kwargs = {
            "vector": query_vector,
            "top_k": self.k,
            "include_metadata": True,
        }

        if self.namespace:
            query_kwargs["namespace"] = self.namespace

        if self.search_filter:
            query_kwargs["filter"] = self.search_filter

        results = self._index.query(**query_kwargs)

        matches = getattr(results, "matches", None)
        if matches is None and isinstance(results, dict):
            matches = results.get("matches", [])

        retrieved_docs = []

        for match in matches:
            metadata = getattr(match, "metadata", None)
            score = getattr(match, "score", None)

            if metadata is None and isinstance(match, dict):
                metadata = match.get("metadata", {})
                score = match.get("score", score)

            metadata = dict(metadata or {})
            text = metadata.pop("text", "")

            metadata["score"] = float(score) if score is not None else None

            retrieved_docs.append(
                Document(
                    page_content=text,
                    metadata=metadata,
                )
            )

        return retrieved_docs


retriever = PineconeRetriever(
    index=index,
    embeddings=embeddings,
    k=5,
    namespace=NAMESPACE,
)

print("Retriever created.")


Retriever created.


In [13]:
# Quick retrieval smoke test
test_query = "Who is Benjamin Graham?"

retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved_docs)} document chunk(s).\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Result {i} ---")
    print(f"Source: {doc.metadata.get('source_file')}")
    print(f"Page: {doc.metadata.get('page')}")
    print(f"Score: {doc.metadata.get('score')}")
    print(doc.page_content[:500])
    print()


Query: Who is Benjamin Graham?
Retrieved 5 document chunk(s).

--- Result 1 ---
Source: theintelligentinvestorbook.pdf
Page: 11.0
Score: 0.750385642
x
A Note About Benjamin Graham
by Jason Zweig
Who was Benjamin Graham, and why should you listen to him?
Graham was not only one of the best investors who ever lived; he was
also the greatest practical investment thinker of all time. Before Graham,
money managers behaved much like a medieval guild, guided largely by
superstition, guesswork, and arcane rituals. Graham’s Security Analysis
was the textbook that transformed this musty circle into a modern pro-
fession.
1
And The Intelligent Investor

--- Result 2 ---
Source: theintelligentinvestorbook.pdf
Page: 12.0
Score: 0.745914
xi A Note About Benjamin Graham
erness—on upper Fifth Avenue. But Ben’s father died in 1903, the
porcelain business faltered, and the family slid haltingly into poverty.
Ben’s mother turned their home into a boardinghouse; then, borrow-
ing money to trade stocks “on

## 10. Create the RAG chain


In [23]:
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0.0,
    max_tokens=600,
)

system_prompt = """
You are a helpful personal finance knowledge assistant.

Answer the user's question using ONLY the retrieved context below.

Rules:
1. If the answer is not supported by the context, say: "I don't know based on the provided documents."
2. Do not use outside knowledge.
3. Keep the answer clear and concise.
4. Cite the source file and page number for every important claim.
5. If the retrieved context is weak or unrelated, say that the documents do not contain enough evidence.
6. Always provide references to the source documents, even if the answer is known without them.

Retrieved context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

document_prompt = PromptTemplate.from_template(
    "Source file: {source_file}\n"
    "Page: {page}\n"
    "Chunk ID: {chunk_id}\n"
    "Content:\n{page_content}"
)

question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_prompt=document_prompt,
)

rag_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=question_answer_chain,
)

print("RAG chain created successfully.")


RAG chain created successfully.


## 11. Ask questions


In [24]:
def ask_rag(question: str, show_sources: bool = True) -> Dict[str, Any]:
    response = rag_chain.invoke({"input": question})

    print("Question:")
    print(question)

    print("\nAnswer:")
    print(response["answer"])

    if show_sources:
        print("\nRetrieved sources:")
        for i, doc in enumerate(response.get("context", []), start=1):
            print(
                f"{i}. {doc.metadata.get('source_file')} | "
                f"page {doc.metadata.get('page')} | "
                f"score={doc.metadata.get('score')}"
            )

    return response


## 11a. Conversational RAG with Follow-up Questions

Now we'll create a conversational version that can handle follow-up questions by maintaining chat history.

In [ ]:
class ConversationalRAG:
    """
    A conversational RAG system that maintains chat history and can handle follow-up questions.
    """
    
    def __init__(self, rag_chain, max_history=5):
        self.rag_chain = rag_chain
        self.max_history = max_history
        self.chat_history = []
    
    def _format_chat_history(self):
        """Format recent chat history for context"""
        if not self.chat_history:
            return ""
        
        history_text = "\n\nRecent conversation history:\n"
        for i, (q, a) in enumerate(self.chat_history[-self.max_history:], 1):
            history_text += f"Q{i}: {q}\n"
            history_text += f"A{i}: {a[:200]}{'...' if len(a) > 200 else ''}\n\n"
        
        return history_text
    
    def ask(self, question: str, show_sources: bool = True) -> Dict[str, Any]:
        """
        Ask a question with conversation history context
        """
        # Add conversation context to the question if there's history
        contextual_question = question
        if self.chat_history:
            contextual_question = f"""
Previous conversation context:
{self._format_chat_history()}

Current question: {question}

Please answer the current question, taking into account the previous conversation context if relevant. If the question refers to something mentioned earlier (like "it", "that", "the company", etc.), use the conversation history to understand what the user is referring to.
"""
        
        # Get response from RAG chain
        response = self.rag_chain.invoke({"input": contextual_question})
        answer = response["answer"]
        
        # Add to chat history
        self.chat_history.append((question, answer))
        
        # Display results
        print("Question:")
        print(question)
        print("\nAnswer:")
        print(answer)
        
        if show_sources:
            print("\nRetrieved sources:")
            for i, doc in enumerate(response.get("context", []), start=1):
                print(
                    f"{i}. {doc.metadata.get('source_file')} | "
                    f"page {doc.metadata.get('page')} | "
                    f"score={doc.metadata.get('score'):.3f}"
                )
        
        return response
    
    def clear_history(self):
        """Clear the conversation history"""
        self.chat_history = []
        print("Conversation history cleared.")
    
    def get_history(self):
        """Get the current chat history"""
        return self.chat_history
    
    def show_history(self):
        """Display the conversation history"""
        if not self.chat_history:
            print("No conversation history yet.")
            return
        
        print("=== Conversation History ===")
        for i, (q, a) in enumerate(self.chat_history, 1):
            print(f"\n--- Exchange {i} ---")
            print(f"Q: {q}")
            print(f"A: {a[:300]}{'...' if len(a) > 300 else ''}")
        print("=" * 30)

# Create conversational RAG instance
conv_rag = ConversationalRAG(rag_chain, max_history=5)

print("Conversational RAG system created!")

### Test Conversational Follow-ups

Let's test the conversational capabilities with some follow-up questions:

In [ ]:
# Start a conversation - First question
conv_rag.ask("What is Warren Buffett's investment philosophy?")

In [ ]:
# Follow-up question - notice how it refers to "his approach"
conv_rag.ask("Can you give me specific examples of how he applies this approach?")

In [ ]:
# View conversation history
conv_rag.show_history()

## 12. Streamlit Conversational Interface

Now let's create a complete Streamlit app with conversational capabilities and the ability to clear the chat history.

**Important:** Run this as a separate Python file, not in the notebook!

In [ ]:
# DO NOT RUN THIS CELL IN THE NOTEBOOK!
# Save this code as 'streamlit_app.py' and run with: streamlit run streamlit_app.py

streamlit_app_code = '''
import streamlit as st
import os
import time
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

# Environment variables
from dotenv import load_dotenv

# Pinecone SDK
from pinecone import Pinecone, ServerlessSpec

# LangChain: document loading and splitting
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain: OpenAI models and embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangChain: custom retriever and RAG chain components
from pydantic import PrivateAttr
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Custom chain wrapper classes for LangChain 1.2.17+
class StuffDocumentsChain:
    """Custom implementation of create_stuff_documents_chain with invoke method"""
    
    def __init__(self, llm, prompt, document_prompt=None):
        self.llm = llm
        self.prompt = prompt
        self.document_prompt = document_prompt
    
    def format_docs(self, docs):
        if self.document_prompt:
            formatted = []
            for doc in docs:
                formatted.append(self.document_prompt.format(**doc.metadata, page_content=doc.page_content))
            return "\\n\\n".join(formatted)
        else:
            return "\\n\\n".join(doc.page_content for doc in docs)
    
    def invoke(self, inputs):
        docs = inputs["context"]
        context = self.format_docs(docs)
        # Create a new dict with the formatted context, avoiding duplicate keys
        prompt_inputs = {k: v for k, v in inputs.items() if k != "context"}
        prompt_inputs["context"] = context
        messages = self.prompt.format_messages(**prompt_inputs)
        response = self.llm.invoke(messages)
        return {"answer": response.content, "context": docs}

class RetrievalChain:
    """Custom implementation of create_retrieval_chain with invoke method"""
    
    def __init__(self, retriever, combine_docs_chain):
        self.retriever = retriever
        self.combine_docs_chain = combine_docs_chain
    
    def invoke(self, inputs):
        # Get the query
        query = inputs["input"]
        
        # Retrieve relevant documents
        docs = self.retriever.invoke(query)
        
        # Pass to the combine chain
        chain_input = {"context": docs, "input": query}
        result = self.combine_docs_chain.invoke(chain_input)
        
        return result

def create_stuff_documents_chain(llm, prompt, document_prompt=None):
    """Custom implementation of create_stuff_documents_chain"""
    return StuffDocumentsChain(llm, prompt, document_prompt)

def create_retrieval_chain(retriever, combine_docs_chain):
    """Custom implementation of create_retrieval_chain"""
    return RetrievalChain(retriever, combine_docs_chain)

class PineconeRetriever(BaseRetriever):
    k: int = 5
    namespace: Optional[str] = None
    search_filter: Optional[Dict[str, Any]] = None

    _index: Any = PrivateAttr()
    _embeddings: Any = PrivateAttr()

    def __init__(
        self,
        index: Any,
        embeddings: OpenAIEmbeddings,
        k: int = 5,
        namespace: Optional[str] = None,
        search_filter: Optional[Dict[str, Any]] = None,
        **kwargs: Any,
    ):
        super().__init__(
            k=k,
            namespace=namespace,
            search_filter=search_filter,
            **kwargs,
        )
        self._index = index
        self._embeddings = embeddings

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        query_vector = self._embeddings.embed_query(query)

        query_kwargs = {
            "vector": query_vector,
            "top_k": self.k,
            "include_metadata": True,
        }

        if self.namespace:
            query_kwargs["namespace"] = self.namespace

        if self.search_filter:
            query_kwargs["filter"] = self.search_filter

        results = self._index.query(**query_kwargs)

        matches = getattr(results, "matches", None)
        if matches is None and isinstance(results, dict):
            matches = results.get("matches", [])

        retrieved_docs = []

        for match in matches:
            metadata = getattr(match, "metadata", None)
            score = getattr(match, "score", None)

            if metadata is None and isinstance(match, dict):
                metadata = match.get("metadata", {})
                score = match.get("score", score)

            metadata = dict(metadata or {})
            text = metadata.pop("text", "")

            metadata["score"] = float(score) if score is not None else None

            retrieved_docs.append(
                Document(
                    page_content=text,
                    metadata=metadata,
                )
            )

        return retrieved_docs

class ConversationalRAG:
    """
    A conversational RAG system that maintains chat history and can handle follow-up questions.
    """
    
    def __init__(self, rag_chain, max_history=5):
        self.rag_chain = rag_chain
        self.max_history = max_history
    
    def _format_chat_history(self, chat_history):
        """Format recent chat history for context"""
        if not chat_history:
            return ""
        
        history_text = "\\n\\nRecent conversation history:\\n"
        for i, (q, a) in enumerate(chat_history[-self.max_history:], 1):
            history_text += f"Q{i}: {q}\\n"
            history_text += f"A{i}: {a[:200]}{\\'...\\'  if len(a) > 200 else \\'\\'}\\n\\n"
        
        return history_text
    
    def ask(self, question: str, chat_history: list) -> Dict[str, Any]:
        """
        Ask a question with conversation history context
        """
        # Add conversation context to the question if there\\'s history
        contextual_question = question
        if chat_history:
            contextual_question = f"""
Previous conversation context:
{self._format_chat_history(chat_history)}

Current question: {question}

Please answer the current question, taking into account the previous conversation context if relevant. If the question refers to something mentioned earlier (like "it", "that", "the company", etc.), use the conversation history to understand what the user is referring to.
"""
        
        # Get response from RAG chain
        response = self.rag_chain.invoke({"input": contextual_question})
        return response

@st.cache_resource
def initialize_rag_system():
    """Initialize the RAG system (cached for performance)"""
    
    # Load environment variables
    load_dotenv("env/api_keys.env")
    
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
    
    if not OPENAI_API_KEY or not PINECONE_API_KEY:
        st.error("Please ensure OPENAI_API_KEY and PINECONE_API_KEY are set in env/api_keys.env")
        st.stop()
    
    # Settings
    EMBEDDING_MODEL = "text-embedding-3-small"
    CHAT_MODEL = "gpt-4o-mini"
    INDEX_NAME = "rag-3-pdf-openai"
    NAMESPACE = "personal-finance-pdfs"
    
    # Initialize OpenAI and Pinecone
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index(INDEX_NAME)
    
    # Create retriever
    retriever = PineconeRetriever(
        index=index,
        embeddings=embeddings,
        k=5,
        namespace=NAMESPACE,
    )
    
    # Create RAG chain
    llm = ChatOpenAI(
        model=CHAT_MODEL,
        temperature=0.0,
        max_tokens=600,
    )
    
    system_prompt = """
You are a helpful personal finance knowledge assistant.

Answer the user\\'s question using ONLY the retrieved context below.

Rules:
1. If the answer is not supported by the context, say: "I don\\'t know based on the provided documents."
2. Do not use outside knowledge.
3. Keep the answer clear and concise.
4. Cite the source file and page number for every important claim.
5. If the retrieved context is weak or unrelated, say that the documents do not contain enough evidence.
6. Always provide references to the source documents, even if the answer is known without them.

Retrieved context:
{context}
"""
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])
    
    document_prompt = PromptTemplate.from_template(
        "Source file: {source_file}\\n"
        "Page: {page}\\n"
        "Chunk ID: {chunk_id}\\n"
        "Content:\\n{page_content}"
    )
    
    question_answer_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=prompt,
        document_prompt=document_prompt,
    )
    
    rag_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=question_answer_chain,
    )
    
    # Create conversational RAG
    conv_rag = ConversationalRAG(rag_chain, max_history=5)
    
    return conv_rag

def main():
    st.set_page_config(
        page_title="Personal Finance RAG Assistant",
        page_icon="💰",
        layout="wide"
    )
    
    st.title("💰 Personal Finance RAG Assistant")
    st.markdown("Ask questions about personal finance and get answers from your documents. I can handle follow-up questions too!")
    
    # Initialize RAG system
    try:
        conv_rag = initialize_rag_system()
    except Exception as e:
        st.error(f"Failed to initialize RAG system: {str(e)}")
        st.stop()
    
    # Initialize session state
    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []
    
    # Sidebar with controls
    with st.sidebar:
        st.header("Chat Controls")
        
        if st.button("🗑️ Clear Conversation", type="secondary"):
            st.session_state.chat_history = []
            st.rerun()
        
        st.divider()
        
        st.header("Instructions")
        st.markdown("""
        - Ask any question about personal finance
        - I can answer follow-up questions that refer to previous answers
        - Use \\'Clear Conversation\\' to start fresh
        - I cite sources for all my answers
        """)
        
        if st.session_state.chat_history:
            st.divider()
            st.header("Conversation Stats")
            st.metric("Messages", len(st.session_state.chat_history))
    
    # Main chat interface
    col1, col2 = st.columns([3, 1])
    
    with col1:
        user_question = st.text_input(
            "Ask a question:",
            placeholder="e.g., What is Warren Buffett\\'s investment philosophy?",
            key="user_input"
        )
    
    with col2:
        ask_button = st.button("Ask", type="primary", use_container_width=True)
    
    # Process question
    if (ask_button or user_question) and user_question.strip():
        with st.spinner("Getting answer..."):
            try:
                response = conv_rag.ask(user_question, st.session_state.chat_history)
                answer = response["answer"]
                sources = response.get("context", [])
                
                # Add to chat history
                st.session_state.chat_history.append((user_question, answer))
                
                # Clear the input
                st.session_state.user_input = ""
                
            except Exception as e:
                st.error(f"Error getting answer: {str(e)}")
                return
    
    # Display chat history
    if st.session_state.chat_history:
        st.divider()
        st.header("Conversation")
        
        for i, (question, answer) in enumerate(reversed(st.session_state.chat_history)):
            message_num = len(st.session_state.chat_history) - i
            
            with st.container():
                # Question
                st.markdown(f"**🙋 Question {message_num}:** {question}")
                
                # Answer
                st.markdown(f"**🤖 Answer:** {answer}")
                
                # Sources (for the most recent question only)
                if i == 0 and \\'response\\' in locals():
                    with st.expander("📚 Sources"):
                        sources = response.get("context", [])
                        for j, doc in enumerate(sources, 1):
                            source_file = doc.metadata.get(\\'source_file\\', \\'Unknown\\')
                            page = doc.metadata.get(\\'page\\', \\'Unknown\\')
                            score = doc.metadata.get(\\'score\\', 0)
                            
                            st.markdown(f"**{j}.** {source_file} (Page {page}) - Relevance: {score:.3f}")
                
                st.divider()
    else:
        # Welcome message
        st.info("👋 Welcome! Ask me any question about personal finance. I can handle follow-up questions and maintain context throughout our conversation.")
        
        # Example questions
        st.subheader("💡 Example Questions:")
        example_questions = [
            "What is Warren Buffett\\'s investment philosophy?",
            "What is a PE ratio?",
            "How do I value a stock?",
            "What is the margin of safety principle?",
            "Should I invest $1000 in stocks? Why?"
        ]
        
        for eq in example_questions:
            if st.button(f"💬 {eq}", key=f"example_{hash(eq)}"):
                st.session_state.user_input = eq
                st.rerun()

if __name__ == "__main__":
    main()
\'''

# Save the Streamlit app to a file
with open("streamlit_app.py", "w") as f:
    f.write(streamlit_app_code)

print("✅ Streamlit app saved as 'streamlit_app.py'")
print("📝 To run the app:")
print("1. Install streamlit: pip install streamlit")
print("2. Run the app: streamlit run streamlit_app.py")
print("3. Open your browser to the provided URL")
print("\n🔥 Features:")
print("- Conversational interface with follow-up questions")
print("- Clear conversation button") 
print("- Source citations")
print("- Example questions for easy start")
print("- Chat history display")

### Quick Test of Notebook Conversational Features

You can also test the conversational features right here in the notebook:

In [ ]:
# Clear conversation history and start fresh
conv_rag.clear_history()

# Test with a new conversation
conv_rag.ask("If I have $500, what would be the best way to start investing?")

In [ ]:
# Follow-up question referring to the previous answer
conv_rag.ask("What are the risks with that approach?")

### Install Streamlit (if not already installed)

Run this cell to install Streamlit for the web interface:

In [ ]:
# Install Streamlit
import subprocess
import sys

try:
    import streamlit
    print("✅ Streamlit is already installed!")
    print(f"Version: {streamlit.__version__}")
except ImportError:
    print("Installing Streamlit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])
    print("✅ Streamlit installed successfully!")

print("\n🚀 To run the Streamlit app:")
print("1. Open a terminal/command prompt")
print("2. Navigate to your project directory")
print("3. Run: streamlit run streamlit_app.py")
print("4. Your browser will open automatically with the app!")

In [25]:
response = ask_rag("I want to invest 1000 dollars in stocks, what would you recommend? and why?")

Question:
I want to invest 1000 dollars in stocks, what would you recommend? and why?

Answer:
Based on the provided documents, it is recommended to invest the majority of your funds in an index fund and limit your individual stock picking to a maximum of 10% of your overall portfolio. Therefore, if you have $1,000 to invest, you should consider putting $900 in an index fund and $100 in individual stocks. This approach allows you to have a solid core investment while still experimenting with stock picking (theintelligentinvestorbook.pdf, p. 381.0; p. 411.0).

Additionally, if you enjoy the stock picking process and earn good returns, you can gradually increase your stock investments, but always keep the majority in an index fund to mitigate risk (theintelligentinvestorbook.pdf, p. 411.0).

Retrieved sources:
1. theintelligentinvestorbook.pdf | page 411.0 | score=0.506824076
2. theintelligentinvestorbook.pdf | page 381.0 | score=0.50576514
3. theintelligentinvestorbook.pdf | page 145.0 

In [26]:
response = ask_rag("Who is Warren Buffett?")

Question:
Who is Warren Buffett?

Answer:
Warren Buffett is described as Graham's greatest student and has become the world’s most successful investor by putting new twists on Benjamin Graham's ideas. He, along with his partner Charles Munger, emphasizes the importance of "margin of safety" and detachment from the market, while also focusing on future growth. Buffett looks for "franchise" companies with strong consumer brands, robust financial health, and near-monopolies in their markets (theintelligentinvestorbook.pdf, p. 415.0).

Retrieved sources:
1. theintelligentinvestorbook.pdf | page 415.0 | score=0.596466362
2. theintelligentinvestorbook.pdf | page 415.0 | score=0.542960167
3. theintelligentinvestorbook.pdf | page 557.0 | score=0.534993589
4. theintelligentinvestorbook.pdf | page 556.0 | score=0.498456895
5. theintelligentinvestorbook.pdf | page 9.0 | score=0.474547565


In [27]:
response = ask_rag("What is a PE ratio and forward PE ratio?")

Question:
What is a PE ratio and forward PE ratio?

Answer:
The price/earnings (P/E) ratio is the current price of a stock divided by the current (or sometimes projected) earnings per share of the issuing company. It is a widely used stock analysis statistic that helps investors assess how cheap or expensive a stock price is. A relatively high P/E ratio generally indicates that investors believe the company's earnings are likely to grow quickly (personalfinancefordummies.pdf, p. 457).

The forward P/E ratio is derived by dividing the current price by projected future earnings. However, it is considered nonsensical by some, as it relies on unknown future earnings rather than known current earnings (theintelligentinvestorbook.pdf, p. 388).

Retrieved sources:
1. personalfinancefordummies.pdf | page 457.0 | score=0.54292655
2. theintelligentinvestorbook.pdf | page 84.0 | score=0.508330286
3. theintelligentinvestorbook.pdf | page 388.0 | score=0.499239743
4. theintelligentinvestorbook.pdf 

In [18]:
response = ask_rag("What is margin of safety?")

Question:
What is margin of safety?

Answer:
The margin of safety is a principle in investing that provides a cushion against errors in judgment or unforeseen market fluctuations. It is the difference between the intrinsic value of an investment and its market price, allowing investors to feel protected against potential losses. A large margin of safety means that future earnings are expected to remain above a certain level, providing a buffer against declines in value. It can be calculated for bonds by comparing the total value of the enterprise with its debt, ensuring that there is enough value to cover potential losses (theintelligentinvestorbook.pdf, page 527.0). The concept emphasizes prudent investment practices, where estimates should err on the side of understatement (theintelligentinvestorbook.pdf, page 531.0).

Retrieved sources:
1. theintelligentinvestorbook.pdf | page 533.0 | score=0.645613313
2. theintelligentinvestorbook.pdf | page 527.0 | score=0.64120847
3. theintellige

In [19]:
response = ask_rag("Who is Nikola Jokic?")

Question:
Who is Nikola Jokic?

Answer:
I don't know based on the provided documents.

Retrieved sources:
1. theintelligentinvestorbook.pdf | page 53.0 | score=0.207077175
2. theintelligentinvestorbook.pdf | page 97.0 | score=0.193330526
3. theintelligentinvestorbook.pdf | page 622.0 | score=0.182083517
4. theintelligentinvestorbook.pdf | page 405.0 | score=0.17677173
5. theintelligentinvestorbook.pdf | page 539.0 | score=0.171681598


## 12. Optional: inspect retrieved context only

Use this when the answer is bad and you want to debug retrieval quality.


In [20]:
def inspect_retrieval(question: str, k: int = 5) -> List[Document]:
    debug_retriever = PineconeRetriever(
        index=index,
        embeddings=embeddings,
        k=k,
        namespace=NAMESPACE,
    )

    docs = debug_retriever.invoke(question)

    for i, doc in enumerate(docs, start=1):
        print(f"\n--- Retrieved chunk {i} ---")
        print(f"Source: {doc.metadata.get('source_file')}")
        print(f"Page: {doc.metadata.get('page')}")
        print(f"Score: {doc.metadata.get('score')}")
        print(doc.page_content[:1200])

    return docs


_ = inspect_retrieval("What are financial ratios?", k=5)



--- Retrieved chunk 1 ---
Source: personalfinancefordummies.pdf
Page: 64.0
Score: 0.445088327
balance sheet, income statement, competitive position, price-earnings 
ratio versus its peer group, and so on?
06_506936-ch02.indd   4006_506936-ch02.indd   40 9/29/09   8:08:46 PM 9/29/09   8:08:46 PM

--- Retrieved chunk 2 ---
Source: personalfinancefordummies.pdf
Page: 451.0
Score: 0.425994545
it is a public company, also has an implicit federal government guarantee that 
was demonstrated in late 2008 during the financial crisis.
financial assets: A property or investment (such as investment real estate or 
a stock, mutual fund, or bond) that is held primarily as an investment to gen-
erate a positive return over time.
financial liabilities: Your outstanding loans and debts. To determine your net 
worth, subtract your financial liabilities from your financial assets.
financial planners (or advisors): A motley crew that professes an ability 
to direct your financial future. Financial planne

## 13. Optional Streamlit app

Do **not** run this inside the notebook directly. Save this code as `app.py` after the notebook works.

```python
import streamlit as st

st.title("Personal Finance RAG Assistant")

user_question = st.text_input("Ask a question about the uploaded PDFs")

if user_question:
    response = rag_chain.invoke({"input": user_question})
    st.write(response["answer"])

    with st.expander("Retrieved sources"):
        for doc in response.get("context", []):
            st.write(doc.metadata)
```

A production Streamlit app should move notebook code into reusable Python modules first.


## 14. Common errors and fixes

### `ModuleNotFoundError: No module named 'langchain_pinecone'`

This fixed notebook does not use `langchain_pinecone`, so you should not need that package.

### Pinecone rename error

If you see:

```text
The official Pinecone python package has been renamed from pinecone-client to pinecone
```

Run the environment repair cell at the top with `RUN_ENV_REPAIR = True`, then restart the kernel.

### Dimension mismatch

If you previously created an index using Hugging Face `all-MiniLM-L6-v2`, that index likely has dimension `384`.

This notebook uses OpenAI `text-embedding-3-small`, which requires dimension `1536`.

Use a fresh index name such as:

```python
INDEX_NAME = "rag-3-pdf-openai"
```

### Missing PDFs

Make sure your PDFs are inside:

```text
data/external/
```
